In [1]:
# Author: Om Mohite

import os
import pandas as pd
from sqlalchemy import URL, create_engine, text
from dotenv import load_dotenv

load_dotenv()

db_password = os.getenv("DB_PASSWORD")
if not db_password:
    raise ValueError("Set DB_PASSWORD in your .env file.")

server_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
)

server_engine = create_engine(server_url, pool_pre_ping=True)

with server_engine.begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS aggregation_test"))

server_engine.dispose()

database_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
    database="aggregation_test",
)

engine = create_engine(database_url, pool_pre_ping=True)

print("Database aggregation_test is ready.")

Database aggregation_test is ready.


In [3]:
# Author: Om Mohite

import re
from pathlib import Path

sql_file = Path.cwd() / "test_case_18.sql"
if not sql_file.exists():
    raise FileNotFoundError(f"SQL file not found: {sql_file}")

sql_script = sql_file.read_text(encoding="utf-8")
load_statements = []

for statement in sql_script.split(";"):
    statement = statement.strip()

    if re.match(
        r"^(CREATE TABLE|INSERT INTO sales\b)",
        statement,
        re.IGNORECASE,
    ):
        load_statements.append(statement)

with engine.begin() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS sales")

    for statement in load_statements:
        conn.exec_driver_sql(statement)

with engine.connect() as conn:
    row_count = conn.execute(
        text("SELECT COUNT(*) FROM sales")
    ).scalar_one()

    date_range = conn.execute(
        text("SELECT MIN(sale_date), MAX(sale_date) FROM sales")
    ).one()

print("SQL file used:", sql_file.name)
print("Sales rows:", row_count)
print("Date range:", date_range[0], "to", date_range[1])

SQL file used: test_case_18.sql
Sales rows: 100000
Date range: 2023-01-01 to 2026-12-31


In [4]:
# Author: Om Mohite

region_summary_query = text("""
    SELECT
        region,
        SUM(total_amount) AS total_revenue,
        SUM(quantity) AS total_quantity,
        ROUND(AVG(total_amount), 2) AS average_sale,
        COUNT(*) AS total_transactions,
        MAX(total_amount) AS highest_sale,
        MIN(total_amount) AS lowest_sale
    FROM sales
    GROUP BY region
    ORDER BY total_revenue DESC
""")

with engine.connect() as conn:
    region_summary = pd.read_sql(region_summary_query, conn)

display(region_summary)

,region,total_revenue,total_quantity,average_sale,total_transactions,highest_sale,lowest_sale
0,Central,52850000.0,150000.0,2642.5,20000,5990.0,520.0
1,West,45690000.0,130000.0,2284.5,20000,5382.0,412.0
2,East,38570000.0,110000.0,1928.5,20000,4776.0,306.0
3,South,31490000.0,90000.0,1574.5,20000,4172.0,202.0
4,North,24450000.0,70000.0,1222.5,20000,3570.0,100.0


In [5]:
# Author: Om Mohite

yearly_region_query = text("""
    SELECT
        region,
        YEAR(sale_date) AS sale_year,
        SUM(total_amount) AS total_revenue
    FROM sales
    GROUP BY region, YEAR(sale_date)
    ORDER BY region, sale_year
""")

with engine.connect() as conn:
    yearly_region_revenue = pd.read_sql(yearly_region_query, conn)

display(yearly_region_revenue)

,region,sale_year,total_revenue
0,Central,2023,13207850.0
1,Central,2024,13359710.0
2,Central,2025,13220170.0
3,Central,2026,13062270.0
4,East,2023,9641712.0
5,East,2024,9747460.0
6,East,2025,9648084.0
7,East,2026,9532744.0
8,North,2023,6111770.0
9,North,2024,6176550.0


In [6]:
# Author: Om Mohite

monthly_region_query = text("""
    SELECT
        region,
        YEAR(sale_date) AS sale_year,
        MONTH(sale_date) AS sale_month,
        SUM(total_amount) AS total_revenue
    FROM sales
    GROUP BY region, YEAR(sale_date), MONTH(sale_date)
    ORDER BY region, sale_year, sale_month
""")

with engine.connect() as conn:
    monthly_region_revenue = pd.read_sql(monthly_region_query, conn)

display(monthly_region_revenue.head(20))

,region,sale_year,sale_month,total_revenue
0,Central,2023,1,1147300.0
1,Central,2023,2,1035670.0
2,Central,2023,3,1129240.0
3,Central,2023,4,1087870.0
4,Central,2023,5,1106300.0
5,Central,2023,6,1064520.0
6,Central,2023,7,1101540.0
7,Central,2023,8,1110190.0
8,Central,2023,9,1075270.0
9,Central,2023,10,1120140.0


In [7]:
# Author: Om Mohite

product_performance_query = text("""
    SELECT
        product_id,
        SUM(quantity) AS total_quantity,
        SUM(total_amount) AS total_revenue
    FROM sales
    GROUP BY product_id
    ORDER BY total_revenue DESC
    LIMIT 10
""")

with engine.connect() as conn:
    top_products = pd.read_sql(product_performance_query, conn)

display(top_products)


,product_id,total_quantity,total_revenue
0,100,10000.0,3990000.0
1,90,10000.0,3890000.0
2,80,10000.0,3790000.0
3,70,10000.0,3690000.0
4,60,10000.0,3590000.0
5,99,9000.0,3582000.0
6,89,9000.0,3492000.0
7,50,10000.0,3490000.0
8,79,9000.0,3402000.0
9,40,10000.0,3390000.0


In [8]:
# Author: Om Mohite

regional_pivot_query = text("""
    SELECT
        SUM(CASE WHEN region = 'North' THEN total_amount ELSE 0 END) AS north_revenue,
        SUM(CASE WHEN region = 'South' THEN total_amount ELSE 0 END) AS south_revenue,
        SUM(CASE WHEN region = 'East' THEN total_amount ELSE 0 END) AS east_revenue,
        SUM(CASE WHEN region = 'West' THEN total_amount ELSE 0 END) AS west_revenue,
        SUM(CASE WHEN region = 'Central' THEN total_amount ELSE 0 END) AS central_revenue
    FROM sales
""")

with engine.connect() as conn:
    regional_pivot = pd.read_sql(regional_pivot_query, conn)

display(regional_pivot)

,north_revenue,south_revenue,east_revenue,west_revenue,central_revenue
0,24450000.0,31490000.0,38570000.0,45690000.0,52850000.0


In [9]:
# Author: Om Mohite

yearly_growth_query = text("""
    WITH yearly_revenue AS (
        SELECT
            YEAR(sale_date) AS sale_year,
            SUM(total_amount) AS total_revenue
        FROM sales
        GROUP BY YEAR(sale_date)
    ),
    revenue_with_previous_year AS (
        SELECT
            sale_year,
            total_revenue,
            LAG(total_revenue) OVER (ORDER BY sale_year) AS previous_year_revenue
        FROM yearly_revenue
    )
    SELECT
        sale_year,
        total_revenue,
        previous_year_revenue,
        ROUND(
            (total_revenue - previous_year_revenue)
            * 100.0 / NULLIF(previous_year_revenue, 0),
            2
        ) AS growth_percentage
    FROM revenue_with_previous_year
    ORDER BY sale_year
""")

with engine.connect() as conn:
    yearly_growth = pd.read_sql(yearly_growth_query, conn)

display(yearly_growth)

,sale_year,total_revenue,previous_year_revenue,growth_percentage
0,2023,48251110.0,NaN,NaN
1,2024,48792870.0,48251110.0,1.12
2,2025,48289390.0,48792870.0,-1.03
3,2026,47716630.0,48289390.0,-1.19


In [10]:
# Author: Om Mohite

index_definitions = {
    "idx_sales_region": "CREATE INDEX idx_sales_region ON sales(region)",
    "idx_sales_date": "CREATE INDEX idx_sales_date ON sales(sale_date)",
    "idx_sales_product": "CREATE INDEX idx_sales_product ON sales(product_id)",
    "idx_sales_customer": "CREATE INDEX idx_sales_customer ON sales(customer_id)",
    "idx_sales_region_date": (
        "CREATE INDEX idx_sales_region_date ON sales(region, sale_date)"
    ),
}

with engine.begin() as conn:
    existing_indexes = {
        row[2]
        for row in conn.exec_driver_sql("SHOW INDEX FROM sales").fetchall()
    }

    for index_name, create_sql in index_definitions.items():
        if index_name not in existing_indexes:
            conn.exec_driver_sql(create_sql)
            print("Created:", index_name)
        else:
            print("Already exists:", index_name)

print("Index setup complete.")

Created: idx_sales_region
Created: idx_sales_date
Created: idx_sales_product
Created: idx_sales_customer
Created: idx_sales_region_date
Index setup complete.


In [11]:
# Author: Om Mohite

explain_aggregation_query = text("""
    EXPLAIN
    SELECT
        region,
        SUM(total_amount) AS total_revenue
    FROM sales
    WHERE sale_date >= '2025-01-01'
      AND sale_date < '2026-01-01'
    GROUP BY region
""")

with engine.connect() as conn:
    aggregation_plan = pd.read_sql(explain_aggregation_query, conn)

display(aggregation_plan)

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,sales,None,index,"idx_sales_region,idx_sales_date,idx_sales_regi...",idx_sales_region,202,None,62562,50.0,Using where


In [12]:
# Author: Om Mohite

with engine.begin() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS sales_partitioned")

    conn.exec_driver_sql("""
        CREATE TABLE sales_partitioned (
            sale_id INT NOT NULL,
            sale_date DATE NOT NULL,
            region VARCHAR(50) NOT NULL,
            product_id INT NOT NULL,
            customer_id INT NOT NULL,
            quantity INT NOT NULL,
            unit_price DECIMAL(10,2) NOT NULL,
            total_amount DECIMAL(12,2) NOT NULL,
            PRIMARY KEY (sale_id, sale_date)
        )
        PARTITION BY RANGE COLUMNS (sale_date) (
            PARTITION p2023 VALUES LESS THAN ('2024-01-01'),
            PARTITION p2024 VALUES LESS THAN ('2025-01-01'),
            PARTITION p2025 VALUES LESS THAN ('2026-01-01'),
            PARTITION p2026 VALUES LESS THAN ('2027-01-01'),
            PARTITION p_future VALUES LESS THAN (MAXVALUE)
        )
    """)

    conn.exec_driver_sql("""
        INSERT INTO sales_partitioned
        SELECT *
        FROM sales
    """)

    partitioned_count = conn.execute(
        text("SELECT COUNT(*) FROM sales_partitioned")
    ).scalar_one()

print("Rows copied to partitioned table:", partitioned_count)

Rows copied to partitioned table: 100000


In [13]:
# Author: Om Mohite

partition_info_query = text("""
    SELECT
        PARTITION_NAME,
        TABLE_ROWS
    FROM INFORMATION_SCHEMA.PARTITIONS
    WHERE TABLE_SCHEMA = DATABASE()
      AND TABLE_NAME = 'sales_partitioned'
    ORDER BY PARTITION_ORDINAL_POSITION
""")

with engine.connect() as conn:
    partition_info = pd.read_sql(partition_info_query, conn)

display(partition_info)

,PARTITION_NAME,TABLE_ROWS
0,p2023,24661
1,p2024,25298
2,p2025,25020
3,p2026,25020
4,p_future,0


In [14]:
# Author: Om Mohite

partitioned_revenue_query = text("""
    SELECT
        region,
        SUM(total_amount) AS total_revenue
    FROM sales_partitioned
    WHERE sale_date >= '2025-01-01'
      AND sale_date < '2026-01-01'
    GROUP BY region
    ORDER BY total_revenue DESC
""")

with engine.connect() as conn:
    partitioned_revenue = pd.read_sql(partitioned_revenue_query, conn)

display(partitioned_revenue)

,region,total_revenue
0,Central,13220170.0
1,West,11428808.0
2,East,9648084.0
3,South,7877068.0
4,North,6115260.0


In [15]:
# Author: Om Mohite

partitioned_explain_query = text("""
    EXPLAIN
    SELECT
        region,
        SUM(total_amount) AS total_revenue
    FROM sales_partitioned
    WHERE sale_date >= '2025-01-01'
      AND sale_date < '2026-01-01'
    GROUP BY region
""")

with engine.connect() as conn:
    partitioned_plan = pd.read_sql(partitioned_explain_query, conn)

display(partitioned_plan)

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,sales_partitioned,p2025,ALL,None,None,None,None,25020,11.11,Using where; Using temporary


In [16]:
# Author: Om Mohite

with engine.begin() as conn:
    for shard_name in ["sales_shard_1", "sales_shard_2", "sales_shard_3"]:
        conn.exec_driver_sql(f"DROP TABLE IF EXISTS {shard_name}")
        conn.exec_driver_sql(
            f"CREATE TABLE {shard_name} LIKE sales"
        )

    conn.exec_driver_sql("""
        INSERT INTO sales_shard_1
        SELECT * FROM sales
        WHERE MOD(sale_id, 3) = 0
    """)

    conn.exec_driver_sql("""
        INSERT INTO sales_shard_2
        SELECT * FROM sales
        WHERE MOD(sale_id, 3) = 1
    """)

    conn.exec_driver_sql("""
        INSERT INTO sales_shard_3
        SELECT * FROM sales
        WHERE MOD(sale_id, 3) = 2
    """)

    shard_counts = {
        shard_name: conn.execute(
            text(f"SELECT COUNT(*) FROM {shard_name}")
        ).scalar_one()
        for shard_name in ["sales_shard_1", "sales_shard_2", "sales_shard_3"]
    }

print("Rows per shard:", shard_counts)
print("Total sharded rows:", sum(shard_counts.values()))

Rows per shard: {'sales_shard_1': 33333, 'sales_shard_2': 33334, 'sales_shard_3': 33333}
Total sharded rows: 100000


In [17]:
# Author: Om Mohite

sharded_monthly_query = text("""
    SELECT
        YEAR(sale_date) AS sale_year,
        MONTH(sale_date) AS sale_month,
        SUM(total_amount) AS total_revenue
    FROM (
        SELECT sale_date, total_amount FROM sales_shard_1
        UNION ALL
        SELECT sale_date, total_amount FROM sales_shard_2
        UNION ALL
        SELECT sale_date, total_amount FROM sales_shard_3
    ) AS combined_sales
    GROUP BY YEAR(sale_date), MONTH(sale_date)
    ORDER BY sale_year, sale_month
""")

with engine.connect() as conn:
    sharded_monthly_revenue = pd.read_sql(sharded_monthly_query, conn)

display(sharded_monthly_revenue.head(20))

,sale_year,sale_month,total_revenue
0,2023,1,4207080.0
1,2023,2,3768642.0
2,2023,3,4135308.0
3,2023,4,3966110.0
4,2023,5,4053280.0
5,2023,6,3877460.0
6,2023,7,4022870.0
7,2023,8,4041248.0
8,2023,9,3935120.0
9,2023,10,4092814.0


In [18]:


top_regions_query = text("""
    SELECT
        region,
        SUM(total_amount) AS total_revenue
    FROM sales
    GROUP BY region
    ORDER BY total_revenue DESC
    LIMIT 5
""")

with engine.connect() as conn:
    top_regions = pd.read_sql(top_regions_query, conn)

display(top_regions)

,region,total_revenue
0,Central,52850000.0
1,West,45690000.0
2,East,38570000.0
3,South,31490000.0
4,North,24450000.0


In [19]:
# Author: Om Mohite

regional_share_query = text("""
    SELECT
        region,
        SUM(total_amount) AS total_revenue,
        ROUND(
            SUM(total_amount) * 100.0
            / (SELECT SUM(total_amount) FROM sales),
            2
        ) AS revenue_percentage
    FROM sales
    GROUP BY region
    ORDER BY revenue_percentage DESC
""")

with engine.connect() as conn:
    regional_revenue_share = pd.read_sql(regional_share_query, conn)

display(regional_revenue_share)

,region,total_revenue,revenue_percentage
0,Central,52850000.0,27.38
1,West,45690000.0,23.67
2,East,38570000.0,19.98
3,South,31490000.0,16.31
4,North,24450000.0,12.67


In [20]:
# Author: Om Mohite

top_products_by_quantity_query = text("""
    SELECT
        product_id,
        SUM(quantity) AS total_quantity
    FROM sales
    GROUP BY product_id
    ORDER BY total_quantity DESC
    LIMIT 10
""")

with engine.connect() as conn:
    top_products_by_quantity = pd.read_sql(
        top_products_by_quantity_query,
        conn,
    )

display(top_products_by_quantity)

,product_id,total_quantity
0,60,10000.0
1,20,10000.0
2,30,10000.0
3,40,10000.0
4,50,10000.0
5,80,10000.0
6,70,10000.0
7,90,10000.0
8,10,10000.0
9,100,10000.0


In [21]:
# Author: Om Mohite

best_month_query = text("""
    WITH monthly_revenue AS (
        SELECT
            YEAR(sale_date) AS sale_year,
            MONTH(sale_date) AS sale_month,
            SUM(total_amount) AS total_revenue
        FROM sales
        GROUP BY YEAR(sale_date), MONTH(sale_date)
    ),
    ranked_months AS (
        SELECT
            sale_year,
            sale_month,
            total_revenue,
            RANK() OVER (
                PARTITION BY sale_year
                ORDER BY total_revenue DESC
            ) AS revenue_rank
        FROM monthly_revenue
    )
    SELECT
        sale_year,
        sale_month,
        total_revenue
    FROM ranked_months
    WHERE revenue_rank = 1
    ORDER BY sale_year
""")

with engine.connect() as conn:
    best_month_by_year = pd.read_sql(best_month_query, conn)

display(best_month_by_year)

,sale_year,sale_month,total_revenue
0,2023,1,4207080.0
1,2024,3,4231250.0
2,2025,8,4173842.0
3,2026,12,4153438.0


In [22]:
# Author: Om Mohite

cumulative_revenue_query = text("""
    SELECT
        sale_date,
        SUM(total_amount) AS daily_revenue,
        SUM(SUM(total_amount)) OVER (
            ORDER BY sale_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue
    FROM sales
    GROUP BY sale_date
    ORDER BY sale_date
""")

with engine.connect() as conn:
    cumulative_revenue = pd.read_sql(cumulative_revenue_query, conn)

display(cumulative_revenue.head(20))

,sale_date,daily_revenue,cumulative_revenue
0,2023-01-01,135870.0,135870.0
1,2023-01-02,139220.0,275090.0
2,2023-01-03,140748.0,415838.0
3,2023-01-04,139544.0,555382.0
4,2023-01-05,135608.0,690990.0
5,2023-01-06,133940.0,824930.0
6,2023-01-07,134540.0,959470.0
7,2023-01-08,131908.0,1091378.0
8,2023-01-09,131544.0,1222922.0
9,2023-01-10,133448.0,1356370.0


In [23]:
# Author: Om Mohite

moving_average_query = text("""
    WITH monthly_revenue AS (
        SELECT
            YEAR(sale_date) AS sale_year,
            MONTH(sale_date) AS sale_month,
            SUM(total_amount) AS monthly_revenue
        FROM sales
        GROUP BY YEAR(sale_date), MONTH(sale_date)
    )
    SELECT
        sale_year,
        sale_month,
        monthly_revenue,
        ROUND(
            AVG(monthly_revenue) OVER (
                ORDER BY sale_year, sale_month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ),
            2
        ) AS three_month_moving_average
    FROM monthly_revenue
    ORDER BY sale_year, sale_month
""")

with engine.connect() as conn:
    monthly_moving_average = pd.read_sql(moving_average_query, conn)

display(monthly_moving_average.head(20))

,sale_year,sale_month,monthly_revenue,three_month_moving_average
0,2023,1,4207080.0,4207080.00
1,2023,2,3768642.0,3987861.00
2,2023,3,4135308.0,4037010.00
3,2023,4,3966110.0,3956686.67
4,2023,5,4053280.0,4051566.00
5,2023,6,3877460.0,3965616.67
6,2023,7,4022870.0,3984536.67
7,2023,8,4041248.0,3980526.00
8,2023,9,3935120.0,3999746.00
9,2023,10,4092814.0,4023060.67


In [24]:
# Author: Om Mohite

high_spending_customers_query = text("""
    WITH customer_revenue AS (
        SELECT
            customer_id,
            SUM(total_amount) AS total_spending
        FROM sales
        GROUP BY customer_id
    )
    SELECT
        customer_id,
        total_spending
    FROM customer_revenue
    WHERE total_spending > (
        SELECT AVG(total_spending)
        FROM customer_revenue
    )
    ORDER BY total_spending DESC
""")

with engine.connect() as conn:
    high_spending_customers = pd.read_sql(
        high_spending_customers_query,
        conn,
    )

display(high_spending_customers)

,customer_id,total_spending
0,500,59900.0
1,1000,59900.0
2,1500,59900.0
3,2000,59900.0
4,2500,59900.0
...,...,...
4195,8884,19320.0
4196,9177,19320.0
4197,9384,19320.0
4198,9677,19320.0


In [25]:
# Author: Om Mohite

base_analyze_query = text("""
    EXPLAIN ANALYZE
    SELECT
        region,
        SUM(total_amount) AS total_revenue
    FROM sales
    WHERE sale_date >= '2025-01-01'
      AND sale_date < '2026-01-01'
    GROUP BY region
""")

partitioned_analyze_query = text("""
    EXPLAIN ANALYZE
    SELECT
        region,
        SUM(total_amount) AS total_revenue
    FROM sales_partitioned
    WHERE sale_date >= '2025-01-01'
      AND sale_date < '2026-01-01'
    GROUP BY region
""")

with engine.connect() as conn:
    base_execution_plan = pd.read_sql(base_analyze_query, conn)
    partitioned_execution_plan = pd.read_sql(partitioned_analyze_query, conn)

print("Base table execution plan:")
display(base_execution_plan)

print("Partitioned table execution plan:")
display(partitioned_execution_plan)

Base table execution plan:


,EXPLAIN
0,-> Group aggregate: sum(sales.total_amount) (...


Partitioned table execution plan:


,EXPLAIN
0,-> Table scan on <temporary> (actual time=439...


In [26]:
# Author: Om Mohite

with engine.connect() as conn:
    base_count = conn.execute(
        text("SELECT COUNT(*) FROM sales")
    ).scalar_one()

    partitioned_count = conn.execute(
        text("SELECT COUNT(*) FROM sales_partitioned")
    ).scalar_one()

    shard_count = conn.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM sales_shard_1) +
            (SELECT COUNT(*) FROM sales_shard_2) +
            (SELECT COUNT(*) FROM sales_shard_3)
    """)).scalar_one()

    base_revenue = conn.execute(
        text("SELECT SUM(total_amount) FROM sales")
    ).scalar_one()

    partitioned_revenue_total = conn.execute(
        text("SELECT SUM(total_amount) FROM sales_partitioned")
    ).scalar_one()

    sharded_revenue_total = conn.execute(text("""
        SELECT SUM(total_amount)
        FROM (
            SELECT total_amount FROM sales_shard_1
            UNION ALL
            SELECT total_amount FROM sales_shard_2
            UNION ALL
            SELECT total_amount FROM sales_shard_3
        ) AS all_shards
    """)).scalar_one()

print("Base table rows:", base_count)
print("Partitioned table rows:", partitioned_count)
print("Sharded rows:", shard_count)
print("Base revenue:", base_revenue)
print("Partitioned revenue:", partitioned_revenue_total)
print("Sharded revenue:", sharded_revenue_total)

assert base_count == 100000
assert partitioned_count == base_count
assert shard_count == base_count
assert base_revenue == partitioned_revenue_total == sharded_revenue_total
assert len(region_summary) == 5

print("TEST CASE 18 AGGREGATION AND SCALING VALIDATION PASSED")

Base table rows: 100000
Partitioned table rows: 100000
Sharded rows: 100000
Base revenue: 193050000.00
Partitioned revenue: 193050000.00
Sharded revenue: 193050000.00
TEST CASE 18 AGGREGATION AND SCALING VALIDATION PASSED
